# Caron

## Todo:
- [x] Create a 3D 1000x1000x100 volume
- [x] Create a mock simulation
- [x] Obtain a 1000x1000x100 numpy array
- [ ] Build a GUI with dearpygui

## Imports

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from matplotlib import cm
import os
import imageio.v2 as imageio
import dearpygui.dearpygui as dpg
import time


## Create a mock simulation set

### Create the frames

In [ ]:
do_it=False

if do_it:
    # Settings
    num_frames = 100
    img_size = 1000

    initial_centre = (img_size // 2, img_size // 2)  # (y, x)
    final_centre = (img_size // 2, 20)  # towards left

    initial_aperture_deg = 30
    final_aperture_deg = 120
    initial_sigma = 0.5
    final_sigma = 10

    # Radial decay settings
    decay_length = img_size / 3  # characteristic decay length

    # Noise settings
    noise_amplitude_signal = 0.15  # standard deviation inside cone
    noise_amplitude_background = 0.15  # background noise standard deviation (same or different)

    # Precompute coordinate grid
    y, x = np.indices((img_size, img_size))

    frames = []

    # --- First: Build first frame to compute initial minimum cone value ---

    # Frame 0 interpolation
    t0 = 0
    aperture_0 = initial_aperture_deg
    sigma_0 = initial_sigma
    centre_y_0 = initial_centre[0]
    centre_x_0 = initial_centre[1]

    x_rel0 = x - centre_x_0
    y_rel0 = y - centre_y_0

    angles0 = np.arctan2(y_rel0, x_rel0)
    r0 = np.sqrt(x_rel0**2 + y_rel0**2)

    angles_deg0 = np.degrees(angles0)
    angles_deg0 = (angles_deg0 + 360) % 360
    angles_deg0[angles_deg0 > 180] -= 360

    half_aperture_0 = aperture_0 / 2
    angular_mask0 = np.zeros_like(angles_deg0, dtype=np.float32)
    angular_mask0[np.abs(angles_deg0) <= half_aperture_0] = 1.0
    edge_distance0 = np.clip(half_aperture_0 - np.abs(angles_deg0), 0, 1)
    angular_mask0 *= edge_distance0
    radial_mask0 = np.exp(-r0 / decay_length)
    mask0 = angular_mask0 * radial_mask0

    blurred0 = gaussian_filter(mask0, sigma=sigma_0)

    # Compute minimum nonzero value inside the cone
    cone_pixels0 = blurred0[blurred0 > 0]
    initial_cone_min = cone_pixels0.min()
    background_base_level = 0.5 * initial_cone_min

    print(f"Initial cone minimum value: {initial_cone_min:.5f}")
    print(f"Background base level: {background_base_level:.5f}")

    # --- Now generate all frames ---

    for i in range(num_frames):
        # Interpolation factor
        t = i / (num_frames - 1)
        
        # Interpolate aperture and sigma
        aperture = initial_aperture_deg + (final_aperture_deg - initial_aperture_deg) * t
        sigma = initial_sigma + (final_sigma - initial_sigma) * t
        
        # Interpolate moving apex position
        current_centre_y = initial_centre[0]
        current_centre_x = initial_centre[1] + (final_centre[1] - initial_centre[1]) * t

        # Relative coordinates
        x_rel = x - current_centre_x
        y_rel = y - current_centre_y

        angles = np.arctan2(y_rel, x_rel)
        r = np.sqrt(x_rel**2 + y_rel**2)

        angles_deg = np.degrees(angles)
        angles_deg = (angles_deg + 360) % 360
        angles_deg[angles_deg > 180] -= 360

        # Angular mask
        half_aperture = aperture / 2
        angular_mask = np.zeros_like(angles_deg, dtype=np.float32)
        angular_mask[np.abs(angles_deg) <= half_aperture] = 1.0
        
        # Smooth edge
        edge_distance = np.clip(half_aperture - np.abs(angles_deg), 0, 1)
        angular_mask *= edge_distance

        # Radial decay
        radial_mask = np.exp(-r / decay_length)

        # Final mask (cone signal)
        mask = angular_mask * radial_mask

        # Blur the cone
        blurred_cone = gaussian_filter(mask, sigma=sigma)

        # Add noise to the cone region
        signal_noise = np.random.normal(loc=0.0, scale=noise_amplitude_signal, size=blurred_cone.shape)
        noisy_cone = blurred_cone + signal_noise
        noisy_cone = np.clip(noisy_cone, 0.0, 1.0)

        # Create background noise
        background_noise = np.random.normal(loc=0.0, scale=noise_amplitude_background, size=noisy_cone.shape)
        background = background_base_level + background_noise
        background = np.clip(background, 0.0, 1.0)

        # Create a mask for cone vs background
        cone_region = (blurred_cone > 0)

        # Combine cone and background
        final_frame = np.where(cone_region, noisy_cone, background)

        frames.append(final_frame)
    frames=np.array(frames)

Initial cone minimum value: 0.00000
Background base level: 0.00000


### Save images to disk

In [ ]:
do_it=False

if do_it:
    # Create output directory if it does not exist
    output_dir = "frames"
    os.makedirs(output_dir, exist_ok=True)
    
    # Load the Inferno colormap
    inferno_cmap = cm.inferno
    
    # Save each frame
    for idx, frame in enumerate(frames):
        filename = os.path.join(output_dir, f"frame_{idx:03d}.png")
        # Apply colormap: map [0,1] --> RGBA with inferno
        frame_coloured = inferno_cmap(frame)  # returns RGBA array
        # Drop alpha channel (keep only RGB)
        frame_rgb = (255 * frame_coloured[..., :3]).astype(np.uint8)
            # Save RGB image
        imageio.imwrite(filename, frame_rgb)


### Save the simulation to disk

In [4]:
do_it=False

if do_it:
    np.save('Simulation.npy', frames)

## Create the GUI

In [ ]:
frames = np.load('Simulation.npy')

running = False
speed = 10  # seconds per frame
last_update_time = time.time()
frame_index = 0

frame = frames[0]
frame_min = np.min(frame)
frame_max = np.max(frame)
frame_norm = (frame - frame_min) / (frame_max - frame_min)
frame_rgb = np.stack((frame_norm,) * 3, axis=-1)
frame_rgb = frame_rgb.astype(np.float32) / np.max(frame_rgb)
frame_flattened = frame_rgb.flatten()

dpg.create_context()
#with dpg.font_registry():
#    big_font = dpg.add_font("C:/Windows/Fonts/BASKVILL.TTF", 20, tag="big_font")
with dpg.font_registry():
    big_font = dpg.add_font("C:/Windows/Fonts/BKANT.TTF", 20, tag="big_font")
with dpg.texture_registry(show=True):
    dpg.add_raw_texture(1000, 1000, default_value=frame_flattened, format=dpg.mvFormat_Float_rgb, tag="frame_tag")

def start_callback():
    global running
    running = True

def stop_callback():
    global running
    running = False

def speed_callback(sender, app_data):
    global speed
    speed = app_data

def update_frame():
    global frame_index, last_update_time, running
    current_time = time.time()
    if (running and (current_time - last_update_time) >= 1 / speed) or frame_index == 0:
        last_update_time = current_time
        frame = frames[frame_index]
        frame_min = np.min(frame)
        frame_max = np.max(frame)
        frame_norm = (frame - frame_min) / (frame_max - frame_min)
        frame_norm = np.clip(frame_norm, 0, 1)
        colormap = cm.inferno
        frame_colored = colormap(frame_norm)  # Returns an RGBA array
        frame_rgb = (frame_colored[..., :3] * 255).astype(np.uint8)
        frame_flattened = frame_rgb.flatten().astype(np.float32) / 255.0
        dpg.set_value("frame_tag", frame_flattened)
        frame_index = (frame_index + 1) % len(frames)
    with dpg.mutex():
        target_frame = dpg.get_frame_count() + 2
        dpg.set_frame_callback(target_frame, update_frame)

with dpg.window(label="Jet Creeper", width=1100, height=1000, no_close=True, no_move=True, no_resize=False):
    dpg.bind_font("big_font")
    with dpg.group(label="Visualizator", horizontal=True):
        with dpg.group(label="Map and slider"):
            dpg.add_slider_int(label="Speed (FPS)", height=40, default_value=10, min_value=1, max_value=20, callback=speed_callback)
            dpg.add_image("frame_tag")
        with dpg.group(label="Start&Stop"):
            dpg.add_button(label="Start", callback=start_callback, width=80, height=200)
            dpg.add_button(label="Stop", callback=stop_callback, width=80, height=200)

dpg.create_viewport(title="Our lovely Caron", width=1100, height=1000)

dpg.setup_dearpygui()

# Start updating frames
update_frame()
    
dpg.show_viewport()
dpg.start_dearpygui()
dpg.destroy_context()



Initial frame: 0 last_update_time 1745888752.6029398
Updating frame
Current time: 1745888752.6029398 Last update time: 1745888752.6029398 Speed: 0.1 time difference: 0.0
Updating frame 0 time difference: 0.0
Running
Updating frame
Current time: 1745888752.7187862 Last update time: 1745888752.6029398 Speed: 0.1 time difference: 0.11584639549255371
Updating frame 1 time difference: 0.11584639549255371
Running
Updating frame
Current time: 1745888752.8511052 Last update time: 1745888752.7187862 Speed: 0.1 time difference: 0.13231897354125977
Updating frame 2 time difference: 0.13231897354125977
Running
Updating frame
Current time: 1745888752.9755273 Last update time: 1745888752.8511052 Speed: 0.1 time difference: 0.12442207336425781
Updating frame 3 time difference: 0.12442207336425781
Running
Updating frame
Current time: 1745888752.8511052 Last update time: 1745888752.7187862 Speed: 0.1 time difference: 0.13231897354125977
Updating frame 2 time difference: 0.13231897354125977
Running
Upda

In [ ]:
dpg.delete_item("frame_tag", children_only=False)
dpg.remove_alias("frame_tag")

SystemError: <built-in function remove_alias> returned a result with an exception set